In [1]:
# First time: pip install numpy sounddevice websocket-client python-dotenv (or use your env manager).
# If you already have a 24kHz mono PCM16 WAV, set USE_MIC=False and point TEST_CASES to your files.
import base64
import io
import json
import os
import wave
from datetime import datetime
from pathlib import Path

import numpy as np
import sounddevice as sd
import websocket
from dotenv import load_dotenv
from IPython.display import Audio, display

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
REALTIME_MODEL = "gpt-realtime"
REALTIME_VOICE = "marin"

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Toggle for mic vs. prerecorded WAV input.
USE_MIC = True
MIC_SECONDS = 5.0
MIC_INPUT_WAV = OUTPUT_DIR / f"mic_input_{RUN_TAG}.wav"

# Test cases: edit these to change instruction or voice per run.
TEST_CASES = [
    {
        "id": "neutral_1",
        "wav": "outputs/realtime_input.wav",
        "instruction": "Talk like yoda",
        "voice": REALTIME_VOICE,
    },
]


## OpenAI Realtime (Speech-to-Speech)

Reference: https://platform.openai.com/docs/guides/realtime-websocket
Voice options: https://platform.openai.com/docs/guides/realtime-conversations#voice-options

Notes:
- Input audio should be 24kHz, mono, PCM16 WAV to match `audio/pcm`.
- Configure tests in the first code cell (`TEST_CASES`, `USE_MIC`, `OUTPUT_DIR`).
- Outputs are saved in `notebooks/outputs/` with a timestamp and a JSON manifest. This folder is gitignored.


In [2]:
def read_pcm16_wav(path: str, sample_rate: int = 24000) -> bytes:
    with wave.open(path, "rb") as wf:
        channels = wf.getnchannels()
        sampwidth = wf.getsampwidth()
        rate = wf.getframerate()
        if channels != 1 or sampwidth != 2 or rate != sample_rate:
            raise ValueError(
                f"Expected mono PCM16 {sample_rate}Hz WAV. Got channels={channels}, "
                f"sampwidth={sampwidth}, rate={rate}."
            )
        return wf.readframes(wf.getnframes())


def pcm16_to_wav_bytes(pcm: bytes, sample_rate: int = 24000, channels: int = 1) -> bytes:
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(pcm)
    return buf.getvalue()


# Record mic audio and save a 24kHz mono PCM16 WAV for the API.
def record_wav(
    seconds: float = 5.0,
    sample_rate: int = 24000,
    output_path: str | None = None,
):
    output_path = output_path or str(MIC_INPUT_WAV)
    print(f"Recording {seconds}s at {sample_rate}Hz...")
    # If this hangs, make sure the kernel has microphone permission.
    audio = sd.rec(int(seconds * sample_rate), samplerate=sample_rate, channels=1, dtype="float32")
    sd.wait()
    pcm = (np.clip(audio, -1, 1) * 32767).astype(np.int16)

    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(pcm.tobytes())

    print(f"Saved {output_path}")


In [3]:
# Send the input WAV to the Realtime WebSocket API and return audio + transcript.
def openai_realtime_s2s(
    input_wav_path: str,
    output_wav_path: str | None = None,
    voice: str = REALTIME_VOICE,
    model: str = REALTIME_MODEL,
    instruction_text: str | None = None,
):
    if not OPENAI_API_KEY:
        raise ValueError("Missing OPENAI_API_KEY in notebooks/.env")

    pcm_in = read_pcm16_wav(input_wav_path)

    url = f"wss://api.openai.com/v1/realtime?model={model}"
    auth = OPENAI_API_KEY.strip().strip("'")
    if not auth.startswith("Bearer "):
        auth = f"Bearer {auth}"

    ws = websocket.create_connection(url, header=[f"Authorization: {auth}"])

    def recv_event():
        return json.loads(ws.recv())

    def wait_for_session_updated():
        while True:
            event = recv_event()
            etype = event.get("type")
            if etype == "session.updated":
                return event
            if etype == "error":
                raise RuntimeError(event.get("error"))

    # Consume initial session.created if present
    try:
        first = recv_event()
        if first.get("type") == "error":
            raise RuntimeError(first.get("error"))
    except Exception:
        pass

    session_update = {
        "type": "session.update",
        "session": {
            "type": "realtime",
            "model": model,
            "output_modalities": ["audio"],
            "audio": {
                "input": {
                    "format": {"type": "audio/pcm", "rate": 24000},
                    "turn_detection": None,
                },
                "output": {
                    "format": {"type": "audio/pcm", "rate": 24000},
                    "voice": voice,
                },
            },
        },
    }
    if instruction_text:
        session_update["session"]["instructions"] = instruction_text

    ws.send(json.dumps(session_update))
    wait_for_session_updated()

    # Clear and send input audio
    ws.send(json.dumps({"type": "input_audio_buffer.clear"}))
    ws.send(
        json.dumps(
            {
                "type": "input_audio_buffer.append",
                "audio": base64.b64encode(pcm_in).decode("ascii"),
            }
        )
    )
    ws.send(json.dumps({"type": "input_audio_buffer.commit"}))

    # Explicit response audio format to avoid schema defaults
    ws.send(
        json.dumps(
            {
                "type": "response.create",
                "response": {
                    "output_modalities": ["audio"],
                    "audio": {"output": {"format": {"type": "audio/pcm", "rate": 24000}}},
                },
            }
        )
    )

    out_pcm = bytearray()
    transcript = None
    while True:
        event = recv_event()
        etype = event.get("type")
        if etype in ("response.output_audio.delta", "response.audio.delta"):
            out_pcm.extend(base64.b64decode(event.get("delta", "")))
        elif etype in ("response.output_audio_transcript.done", "response.audio_transcript.done"):
            transcript = event.get("transcript")
        elif etype == "response.done":
            break
        elif etype == "error":
            raise RuntimeError(event.get("error"))

    ws.close()

    wav_bytes = pcm16_to_wav_bytes(bytes(out_pcm))
    if output_wav_path:
        path = Path(output_wav_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(wav_bytes)

    return wav_bytes, transcript


In [4]:
def run_cases():
    if USE_MIC:
        record_wav(seconds=MIC_SECONDS, output_path=str(MIC_INPUT_WAV))

    results = []
    for case in TEST_CASES:
        case_id = case.get("id")
        if not case_id:
            raise ValueError("Each test case needs an 'id'.")

        input_wav = str(MIC_INPUT_WAV) if USE_MIC else case.get("wav")
        if not input_wav:
            raise ValueError(f"Missing 'wav' for case {case_id} (or set USE_MIC=True).")

        instruction = case.get("instruction")
        voice = case.get("voice", REALTIME_VOICE)
        out_wav = OUTPUT_DIR / f"{case_id}_{RUN_TAG}_openai.wav"

        audio_bytes, transcript = openai_realtime_s2s(
            input_wav_path=str(input_wav),
            output_wav_path=str(out_wav),
            voice=voice,
            model=REALTIME_MODEL,
            instruction_text=instruction,
        )

        manifest = {
            "case_id": case_id,
            "timestamp": datetime.now().isoformat(),
            "provider": "openai",
            "model": REALTIME_MODEL,
            "voice": voice,
            "instruction": instruction,
            "input_wav": str(input_wav),
            "output_wav": str(out_wav),
            "transcript": transcript,
        }
        manifest_path = OUTPUT_DIR / f"{case_id}_{RUN_TAG}_openai.json"
        manifest_path.write_text(json.dumps(manifest, indent=2))

        print(f"{case_id}: wrote {out_wav} and {manifest_path}")
        results.append(
            {
                "case_id": case_id,
                "audio_bytes": audio_bytes,
                "transcript": transcript,
                "manifest_path": str(manifest_path),
                "output_wav": str(out_wav),
            }
        )

    return results


results = run_cases()


Recording 5.0s at 24000Hz...
Saved outputs/mic_input_20260204_143254.wav
neutral_1: wrote outputs/neutral_1_20260204_143254_openai.wav and outputs/neutral_1_20260204_143254_openai.json


In [5]:
# Play the last generated audio.
if results:
    display(Audio(results[-1]["audio_bytes"]))
